# Build a data analysis agent

> Build an agent that analyzes data files, generates visualizations, and shares results

## Overview

This guide demonstrates how to build a data analysis agent using a [deep agent](/oss/python/deepagents). Data analysis tasks typically require planning, code execution, and working with artifacts such as scripts, reports, and plots—capabilities that deep agents are designed to handle.

The agent we'll build will:

1. Accept a CSV file for analysis
2. Perform exploratory data analysis and generate visualizations
3. Share results to a Slack channel

<Tip>
  The Slack integration is optional. The agent can be modified to save artifacts locally or share results through other channels.
</Tip>

### Key concepts

This tutorial covers:

* [Backends](/oss/python/deepagents/backends) for sandboxed code execution
* Custom [tools](/oss/python/langchain/tools) for external integrations

## Main Documentation : https://docs.langchain.com/oss/python/deepagents/data-analysis

pip install deepagents

Optional dependencies
For this tutorial, we’ll use:

    Slack Python SDK for sharing results (token setup)
    A LangSmith sandbox for code execution

pip install "langsmith[sandbox]" slack-sdk


Import .env file

In [41]:
import os
from dotenv import load_dotenv
load_dotenv()
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
print(os.getenv("DAYTONA_API_KEY"))

dtn_234c13494eaf3b203df9a8883956e8fa83ac74f24b8a748268b4cff1ef507e62


Set Up the Backend

In [43]:
!uv add langchain-daytona

Resolved 205 packages in 10ms
Checked 197 packages in 21ms


In [35]:
### Sandbox is like a virtual machine given to the Agent , its own runtime, its own file system, isolated from your file system.



## Your virtual fileSystem is ready

## Upload some data for analysis to the virtual backend.
## 

In [ ]:
from daytona_sandbox import create_sandbox, write_csv,cleanup_stale_sandboxes

cleaned = cleanup_stale_sandboxes()
print(f"Cleaned {cleaned} stale sandboxes")
backend = create_sandbox()

result = write_csv(
    backend=backend,
    data=[
        ["Date", "Product", "Units Sold", "Revenue"],
    ["2025-08-01", "Widget A", 10, 250],
    ["2025-08-02", "Widget B", 5, 125],
    ["2025-08-03", "Widget A", 7, 175],
    ["2025-08-04", "Widget C", 3, 90],
    ["2025-08-05", "Widget B", 8, 200],
    ],
    path="/tmp/sales_data_new.csv",
)

Cleaned 0 stale sandboxes


In [ ]:
print(result)

{'status': 'success', 'path': '/tmp/sales_data_new.csv', 'rows_written': 6, 'lines_on_sandbox': 6, 'content_match': True, 'error': None}


### Data analysis tasks might produce artifacts, like reports or plots. The following simple tool downloads them with backend.download_files and then uploads them using the Slack SDK. We could also ask our agent to list the relevant file paths instead of uploading them, so interested parties can obtain them separately as needed.

In [44]:
from langchain.tools import tool
from slack_sdk import WebClient

In [45]:
slack_token = os.environ["SLACK_USER_TOKEN"]
slack_client = WebClient(token=slack_token)

In [57]:
%%writefile analysis_tools.py
"""Additional Tools for analysis"""

@tool(parse_docstring=True)
def slack_send_msg(
    text: str,
    file_path:str|None = None
)-> str:
    """Send message, optionally including attachments such as images.

    Args:
        text: (str) text content of the message
        file_path: (str) file path of attachment in the filesystem.
    """
    # if file path is none
    channel="C0BA8DU1ELV"
    if not file_path:
        slack_client.chat_postMessage(channel=channel,text=text)
    else:
        fp = backend.download_files(
            [file_path]
        )
        slack_client.files_upload_v2(
            channel=channel,
            content=fp[0].content,
            initial_comment=text
        )
        
    return "Message Sent"


Writing analysis_tools.py


In [58]:
%%writefile analysis_agent.py

from deepagents import create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.utils.uuid import uuid7
from langchain.chat_models import init_chat_model
from analysis_tools import slack_send_msg

checkpointer = InMemorySaver()


llm_google = init_chat_model(
    model="google_genai:gemini-3.5-flash"
)

analysis_agent = create_deep_agent(
    model = llm_google,
    tools=[slack_send_msg],
    backend=backend,
    checkpointer=checkpointer
)

thread_id = str(uuid7())
config = {
    "configurable":{
        "thread_id":thread_id
    }
}

from langchain_core.messages import HumanMessage
stream = analysis_agent.stream_events(
    {
        "messages":[
            HumanMessage(
                "Analyze /tmp/sales_data_new.csv in the current dir and generate a beautiful plot. "
        "When finished, send your analysis and the plot to Slack using the tool."
            )
        ]
    },
    version="v3",
    config=config
)

for snapshot in stream.values:
    snapshot["messages"][-1].pretty_print()

Writing analysis_agent.py


sk-or


================================ Human Message =================================

Analyze /tmp/sales_data_new.csv in the current dir and generate a beautiful plot. When finished, send your analysis and the plot to Slack using the tool.
================================ Human Message =================================

Analyze /tmp/sales_data_new.csv in the current dir and generate a beautiful plot. When finished, send your analysis and the plot to Slack using the tool.
================================== Ai Message ==================================

[{'type': 'tool_call', 'id': 'aj4batbl', 'name': 'write_todos', 'args': {'todos': [{'content': 'Locate and read sales data file', 'status': 'completed'}, {'status': 'completed', 'content': 'Analyze data and plan visualization'}, {'content': 'Generate a beautiful plot and save it to a file', 'status': 'completed'}, {'content': 'Send analysis and the plot to Slack', 'status': 'completed'}]}}]
Tool Calls:
  write_todos (aj4batbl)
 Call ID: aj4ba